# Práctica 2 · Memoria y cuantización

**20–25 minutos · Google Colab con GPU NVIDIA · GDG Valladolid**

Compararemos **FP16 y 4 bits NF4** de **SmolLM3-3B**, el mismo modelo de la primera práctica. Usaremos los mismos prompts, una versión fija del checkpoint y generación sin muestreo.

1. Estima cuánto ocuparían sus pesos.
2. Mide memoria y tiempo en las dos versiones.
3. Compara sus respuestas y decide si el ahorro te compensa.

Abre un **entorno nuevo**: no hace falta ejecutar la Práctica 1. En Colab, selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. La primera descarga ocupa varios GB y puede tardar; no cuenta como tiempo de inferencia. No necesitas una API de pago ni iniciar sesión en Hugging Face.

## 1. Preparar el entorno

Instalamos versiones concretas para repetir el ejercicio. Conservamos el PyTorch de Colab. Si acabas de cambiar dependencias que ya habías importado, reinicia la sesión antes de continuar.

In [ ]:
%pip -q install "transformers==4.57.1" "accelerate==1.10.1" "bitsandbytes==0.48.1" pandas matplotlib

In [ ]:
import gc
import json
import time
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from accelerate import init_empty_weights
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig

if not torch.cuda.is_available():
    raise RuntimeError("Esta comparación necesita una GPU NVIDIA. Actívala en Colab y vuelve a ejecutar desde el principio.")
if torch.cuda.get_device_capability(0) < (6, 0):
    raise RuntimeError("La GPU no cumple el requisito mínimo de NF4. Cambia el entorno; no compararemos dispositivos distintos.")
device = torch.device("cuda:0")
torch.cuda.set_device(device)
torch.cuda.init()
HARDWARE = torch.cuda.get_device_name(0)
VERSIONES = {p: version(p) for p in ("torch", "transformers", "accelerate", "bitsandbytes")}
print("GPU:", HARDWARE)
print("VRAM anunciada por CUDA:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GiB")
print("CUDA de PyTorch:", torch.version.cuda)
print("Versiones:", VERSIONES)

## 2. Descargar una vez y preparar los prompts

El modelo tiene licencia **Apache 2.0** ([ficha](https://huggingface.co/HuggingFaceTB/SmolLM3-3B)). Las dos variantes salen de los **mismos archivos**. La cuantización se hace al cargar; no estamos entrenando.

Puedes cambiar los prompts antes de ejecutar. La evaluación es secuencial, con una petición activa; no es una prueba de concurrencia.

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolLM3-3B"
REVISION = "a07cc9a04f16550a088caea529712d1d335b0ac1"
REPETICIONES = 3
MAX_NUEVOS_TOKENS = 64
LIMITE_PRACTICA = 4096  # Límite didáctico, no el máximo real del modelo.
PROMPTS = {
    "Explicación": "Explica en español qué es una API en dos frases y da un ejemplo.",
    "Extracción": 'Devuelve solo JSON con nombre y ciudad: "Lucía vive en Valladolid".',
    "Cálculo": "Una caja contiene 6 paquetes de 4 lápices. ¿Cuántos lápices hay? Responde en una frase en español.",
}

carpeta = snapshot_download(
    MODEL_ID, revision=REVISION, token=False, max_workers=2,
    allow_patterns=["*.json", "*.safetensors", "*.model", "*.jinja", "*.txt"],
)
tokenizer = AutoTokenizer.from_pretrained(carpeta, local_files_only=True)
config_modelo = AutoConfig.from_pretrained(carpeta, local_files_only=True)
entradas_cpu = {}
for caso, prompt in PROMPTS.items():
    texto = tokenizer.apply_chat_template(
        [{"role": "system", "content": "Responde en español de forma breve."},
         {"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    entradas = tokenizer(texto, return_tensors="pt", add_special_tokens=False)
    n = entradas["input_ids"].shape[-1]
    if n + MAX_NUEVOS_TOKENS > min(LIMITE_PRACTICA, config_modelo.max_position_embeddings):
        raise ValueError(f"{caso}: {n} tokens de entrada. Acorta el prompt o revisa el presupuesto de contexto.")
    entradas_cpu[caso] = entradas
    print(f"{caso}: {n} tokens de entrada; hasta {MAX_NUEVOS_TOKENS} de salida.")

## 3. Estimar los pesos antes de cargarlos

**Parámetros × bytes por parámetro.** Contamos los parámetros sin reservar sus tensores: el modelo temporal usa el dispositivo `meta`.

La columna de 4 bits es un **límite ideal**: NF4 añade escalas y metadatos, y deja algunos módulos sin cuantizar. La KV cache, las activaciones y el motor van aparte. **GB = 10⁹ bytes; GiB = 2³⁰ bytes.**

In [ ]:
with init_empty_weights():
    modelo_meta = AutoModelForCausalLM.from_config(config_modelo)
modelo_meta.tie_weights()  # Cuenta una sola vez los pesos compartidos.
N_PARAMETROS = sum(p.numel() for p in modelo_meta.parameters())
del modelo_meta
ideales = pd.DataFrame([
    {"variante": nombre, "GB_pesos_ideales": N_PARAMETROS * b / 1e9,
     "GiB_pesos_ideales": N_PARAMETROS * b / 2**30}
    for nombre, b in [("FP16", 2), ("NF4 · 4 bits", 0.5)]
]).set_index("variante")
print(f"Parámetros: {N_PARAMETROS:,}")
display(ideales.round(3))

## 4. Medir una variante cada vez

Cargamos, calentamos, medimos y liberamos antes de cargar la otra. La versión NF4 usa **pesos de capas lineales en 4 bits**, pero calcula en FP16. No cuantizamos la KV cache.

| Medida | Qué incluye |
|---|---|
| Modelo cargado | Memoria asignada por PyTorch tras cargar, antes del calentamiento. |
| Pico de carga | Máximo asignado durante la carga, no la descarga. |
| Pico al generar | Modelo, KV, activaciones y buffers de esa generación. |
| Reservada | Memoria retenida por el asignador de PyTorch; no toda está en uso. |
| Tiempo de generación | `generate()` sincronizado con CUDA: prefill y decode. Excluye descarga, carga, tokenización y decodificación a texto. |

La memoria CUDA de PyTorch **no es toda la VRAM que muestra `nvidia-smi`**. Indicamos la base de cada carga y su incremento para no confundirlo con memoria exclusiva de pesos.

In [ ]:
def sincronizar():
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def limpiar_gpu():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
        sincronizar()


def memoria():
    if device.type != "cuda":
        return {k: None for k in ("asignada_GiB", "reservada_GiB", "pico_asignado_GiB", "pico_reservado_GiB")}
    return {
        "asignada_GiB": torch.cuda.memory_allocated(device) / 2**30,
        "reservada_GiB": torch.cuda.memory_reserved(device) / 2**30,
        "pico_asignado_GiB": torch.cuda.max_memory_allocated(device) / 2**30,
        "pico_reservado_GiB": torch.cuda.max_memory_reserved(device) / 2**30,
    }


def generar(modelo, caso, max_nuevos):
    entradas = {k: tensor.to(device) for k, tensor in entradas_cpu[caso].items()}
    opciones = GenerationConfig.from_model_config(modelo.config)
    opciones.eos_token_id = modelo.generation_config.eos_token_id
    opciones.pad_token_id = tokenizer.pad_token_id
    sincronizar()
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    inicio = time.perf_counter()
    with torch.inference_mode():
        salida = modelo.generate(
            **entradas, generation_config=opciones, do_sample=False,
            max_new_tokens=max_nuevos, use_cache=True,
        )
    sincronizar()
    segundos = time.perf_counter() - inicio
    ocupacion = memoria()
    n_entrada = entradas["input_ids"].shape[-1]
    tokens_salida = salida.shape[-1] - n_entrada
    respuesta = tokenizer.decode(salida[0, n_entrada:].cpu(), skip_special_tokens=True)
    return {
        "tokens_entrada": n_entrada, "tokens_salida": tokens_salida,
        "generacion_s": segundos, "tokens_s": tokens_salida / segundos,
        "pico_generacion_GiB": ocupacion["pico_asignado_GiB"],
        "pico_reservado_GiB": ocupacion["pico_reservado_GiB"], "respuesta": respuesta,
    }

In [ ]:
def ejecutar_variante(variante):
    modelo = None
    filas = []
    carga = {"variante": variante, "estado": "ERROR"}
    try:
        limpiar_gpu()
        base = memoria()["asignada_GiB"]
        torch.cuda.reset_peak_memory_stats(device)
        opciones = dict(
            local_files_only=True, device_map={"": 0}, dtype=torch.float16,
            low_cpu_mem_usage=True, attn_implementation="sdpa", use_safetensors=True,
        )
        if variante == "NF4 · 4 bits":
            opciones["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
            )
        inicio = time.perf_counter()
        modelo = AutoModelForCausalLM.from_pretrained(carpeta, **opciones).eval()
        sincronizar()
        tiempo_carga = time.perf_counter() - inicio
        if any(p.device.type != "cuda" for p in modelo.parameters()):
            raise RuntimeError("Hay parámetros fuera de CUDA. No es una comparación en la misma GPU.")
        if variante == "NF4 · 4 bits" and not getattr(modelo, "is_loaded_in_4bit", False):
            raise RuntimeError("El modelo no se cargó realmente en 4 bits.")
        capas_4bits = sum(type(m).__name__ == "Linear4bit" for m in modelo.modules())
        if variante == "NF4 · 4 bits" and capas_4bits == 0:
            raise RuntimeError("No se encontró ninguna capa lineal cuantizada.")
        m = memoria()
        carga.update(
            capas_lineales_4bits=capas_4bits,
            base_GiB=base, modelo_cargado_GiB=m["asignada_GiB"],
            incremento_carga_GiB=m["asignada_GiB"] - base,
            pico_carga_GiB=m["pico_asignado_GiB"], carga_s=tiempo_carga,
        )
        # Cada prompt se calienta; el calentamiento no entra en las medidas.
        for caso in PROMPTS:
            generar(modelo, caso, 8)
        for caso in PROMPTS:
            for repeticion in range(1, REPETICIONES + 1):
                filas.append({"variante": variante, "caso": caso,
                              "repeticion": repeticion, **generar(modelo, caso, MAX_NUEVOS_TOKENS)})
        carga["estado"] = "OK"
    except Exception as error:
        carga["error"] = f"{type(error).__name__}: {error}"
        print(f"{variante}: {carga['error']}")
        # No mezclamos un ensayo incompleto con una comparación válida.
        filas = []
    finally:
        modelo = None
        limpiar_gpu()
    return carga, filas

## 5. Ejecutar la comparación

Las dos cargas pueden tardar. Si hay un error, quedará registrado: **no lo sustituimos por un cero ni cambiamos a CPU**. Si una variante falla, no se puede concluir que la otra gana en memoria o velocidad.

In [ ]:
cargas, filas = [], []
for variante in ("FP16", "NF4 · 4 bits"):
    print("Midiendo:", variante)
    carga, medidas = ejecutar_variante(variante)
    cargas.append(carga)
    filas.extend(medidas)

cargas_df = pd.DataFrame(cargas)
resultados = pd.DataFrame(filas)
display(cargas_df.round(3))
cargas_df.to_csv("memoria_cargas.csv", index=False)  # Conserva los errores incluso si fallan ambas cargas.
if resultados.empty:
    raise RuntimeError("No hay medidas válidas. Revisa la GPU y los errores de carga. Si falta memoria, usa un entorno nuevo y cierra otros modelos.")
display(resultados.drop(columns="respuesta").round(3))

## 6. Ver qué cambia

Las barras salen de tus medidas, no de cifras precargadas. Compara cada prompt con el mismo prompt. El límite de salida es igual, pero el modelo puede terminar antes: **lee también los tokens de salida**. Tres repeticiones sirven para practicar; no validan un SLO.

In [ ]:
resumen = resultados.groupby(["variante", "caso"], sort=False).agg(
    generacion_s=("generacion_s", "median"), tokens_s=("tokens_s", "median"),
    tokens_salida=("tokens_salida", "median"), pico_generacion_GiB=("pico_generacion_GiB", "max"),
)
display(resumen.round(3))
validas = cargas_df[cargas_df["estado"] == "OK"].set_index("variante")
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
validas[["modelo_cargado_GiB", "pico_carga_GiB"]].plot.bar(ax=axes[0], rot=0, color=["#485dc0", "#b8c5e6"])
axes[0].set(title="Carga · memoria asignada por PyTorch", xlabel="", ylabel="GiB")
axes[0].legend(["Modelo cargado", "Pico al cargar"])
resumen["generacion_s"].unstack("variante").plot.bar(ax=axes[1], rot=0)
axes[1].set(title="Tiempo de generación · mediana", xlabel="Prompt", ylabel="Segundos")
resumen["pico_generacion_GiB"].unstack("variante").plot.bar(ax=axes[2], rot=0)
axes[2].set(title="Pico al generar · asignado por PyTorch", xlabel="Prompt", ylabel="GiB")
fig.tight_layout()
plt.show()
if set(validas.index) == {"FP16", "NF4 · 4 bits"}:
    ahorro = 1 - validas.loc["NF4 · 4 bits", "incremento_carga_GiB"] / validas.loc["FP16", "incremento_carga_GiB"]
    print(f"Reducción observada del incremento de memoria al cargar: {ahorro:.1%}")
else:
    print("Solo hay una variante completa: falta la otra para comparar.")

## 7. Leer las respuestas

La diferencia entre dos números no dice por sí sola si la respuesta es útil. Revisa idioma, exactitud y cumplimiento de la instrucción. En extracción, comprueba el JSON; en cálculo, la operación. No hay un porcentaje de calidad calculado automáticamente.

In [ ]:
for caso, prompt in PROMPTS.items():
    print(f"\n{caso}: {prompt}")
    for variante in ("FP16", "NF4 · 4 bits"):
        muestra = resultados[(resultados["variante"] == variante) & (resultados["caso"] == caso)]
        print(f"\n{variante}:")
        print(muestra.iloc[0]["respuesta"] if not muestra.empty else "Sin resultado válido; revisa el error.")

## 8. Tu prueba y tus conclusiones

- Alarga uno de los prompts con más contexto y repite desde la sección 2, manteniendo lo demás. ¿Sube el pico de memoria? Los pesos no cambian; la KV y el trabajo de prefill sí pueden crecer.
- ¿Coincide la memoria observada con «parámetros × bytes»? Explica qué falta en esa cuenta.
- Repite también invirtiendo el orden de las variantes antes de sacar conclusiones sobre velocidad.
- ¿La versión de 4 bits conserva las respuestas que necesitas? ¿Ha mejorado el tiempo en **tu GPU**?

Los 4 bits no garantizan más velocidad ni que un modelo quepa. Escalas, módulos sin cuantizar, arquitectura, contexto, kernels y entorno influyen. Estas pruebas no miden concurrencia ni capacidad de producción.

## 9. Guardar las medidas

Descarga los archivos desde el panel **Archivos** de Colab. Incluyen el modelo, la revisión, la GPU y las versiones para poder repetir la prueba. Los errores de carga se conservan en su CSV.

In [ ]:
metadatos = {
    "modelo": MODEL_ID, "revision": REVISION, "gpu": HARDWARE,
    "cuda": torch.version.cuda, "versiones": VERSIONES,
    "parametros": N_PARAMETROS, "repeticiones": REPETICIONES,
    "max_nuevos_tokens": MAX_NUEVOS_TOKENS, "limite_practica": LIMITE_PRACTICA,
    "prompts": PROMPTS, "do_sample": False, "calculo": "FP16", "cache": "sin cuantizar",
    "nf4_double_quant": True, "atencion": "sdpa", "peticiones_activas": 1,
}
resultados.drop(columns="respuesta").to_csv("memoria_medidas.csv", index=False)
cargas_df.to_csv("memoria_cargas.csv", index=False)
Path("memoria_respuestas.json").write_text(
    resultados[["variante", "caso", "repeticion", "respuesta"]].to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8",
)
Path("memoria_entorno.json").write_text(json.dumps(metadatos, ensure_ascii=False, indent=2), encoding="utf-8")
print("Guardados: memoria_medidas.csv, memoria_cargas.csv, memoria_respuestas.json y memoria_entorno.json")

### Si algo falla

- **Sin GPU:** cambia el tipo de entorno y empieza de nuevo. No compares FP16 en CPU con NF4 en GPU.
- **Sin memoria:** reinicia el entorno para liberar modelos anteriores. La tabla conserva el error si falla una variante. No mantengas otras cargas en la misma sesión.
- **Error de bitsandbytes/CUDA:** comprueba la GPU y las versiones impresas. Reinicia tras instalar; consulta la compatibilidad de la versión indicada. No fuerces una carga sin cuantización para completar la tabla.

### Referencias

- [SmolLM3-3B: licencia, arquitectura y plantilla de chat](https://huggingface.co/HuggingFaceTB/SmolLM3-3B).
- [Transformers 4.57.1: NF4, dtype de cálculo y módulos sin cuantizar](https://huggingface.co/docs/transformers/v4.57.1/quantization/bitsandbytes).
- [Compatibilidad de bitsandbytes 0.48.1](https://huggingface.co/docs/bitsandbytes/v0.48.1/installation).
- [PyTorch: memoria CUDA](https://docs.pytorch.org/docs/stable/notes/cuda.html#memory-management).

Este cuaderno no contiene mediciones de ejemplo ni resultados de capacidad validados. Las salidas se generan al ejecutarlo.